# VAEs and GANs: Two Ways to Sample What You Can't Write Down

The runnable code of [VAEs and GANs: Two Ways to Sample What You Can't Write Down](https://residual-stream.lvtan71.workers.dev/posts/vaes-and-gans/), in the order the post runs it. Each
code cell is followed by the output the post printed, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/vaes-and-gans.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 3 min end to end.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
# Setup: fetch the helpers and data this post's code uses into a working folder.
import os, shutil, subprocess, sys
SLUG = "vaes-and-gans"
ROOT = os.path.abspath("..") if os.path.isdir("../shared") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
WORK = os.path.join(ROOT, "work", SLUG)
os.makedirs(WORK, exist_ok=True)
for f in ['shared/vaes-and-gans/gen2d.py']:
    shutil.copy(os.path.join(ROOT, f), WORK)
    if f.endswith(".tar.gz"):                       # the blog's posts, as of that run
        shutil.unpack_archive(os.path.join(WORK, os.path.basename(f)), WORK)
os.chdir(WORK)
sys.path.insert(0, WORK)
print("working in", WORK)

## Checking the bound exactly

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/vaes-and-gans/#checking-the-bound-exactly)

In [ ]:
import numpy as np

# A linear-Gaussian latent model in 2-D, small enough to solve exactly:
#   p(z) = N(0, I),   p(x | z) = N(W z + b, s^2 I).
# Then p(x) = N(b, W W^T + s^2 I), and the true posterior p(z | x) is Gaussian.
# For any Gaussian q(z | x) the ELBO is reconstruction - KL(q || prior), and
# log p(x) - ELBO = KL(q || true posterior). Checked here three ways.
rng = np.random.default_rng(0)
W, b, s = np.array([[1.0, 0.5], [-0.3, 0.8]]), np.array([0.2, -0.1]), 0.5
x = np.array([1.0, 0.4])

def log_normal(v, mean, cov):
    d = v - mean
    return -0.5 * (len(v) * np.log(2 * np.pi) + np.linalg.slogdet(cov)[1] + d @ np.linalg.solve(cov, d))

def kl_gauss(m1, S1, m0, S0):                       # KL(N(m1, S1) || N(m0, S0))
    S0i = np.linalg.inv(S0)
    return 0.5 * (np.trace(S0i @ S1) + (m0 - m1) @ S0i @ (m0 - m1) - len(m1)
                  + np.linalg.slogdet(S0)[1] - np.linalg.slogdet(S1)[1])

def elbo(mu, sigma, n=200_000):
    """Reconstruction term by sampling (the reparameterization trick: z = mu +
    sigma * eps), KL to the prior in closed form."""
    z = mu + sigma * rng.standard_normal((n, 2))
    r = x - z @ W.T - b
    rec = (-0.5 * (r ** 2).sum(1) / s ** 2 - np.log(2 * np.pi * s ** 2)).mean()
    kl = 0.5 * (mu ** 2 + sigma ** 2 - 1 - np.log(sigma ** 2)).sum()
    r0 = x - W @ mu - b                               # the same expectation, in closed form
    exact = -0.5 * (r0 @ r0 + (W ** 2 * sigma ** 2).sum()) / s ** 2 - np.log(2 * np.pi * s ** 2)
    return rec, exact, kl

log_px = log_normal(x, b, W @ W.T + s ** 2 * np.eye(2))
P = np.linalg.inv(np.eye(2) + W.T @ W / s ** 2)      # posterior covariance
m = P @ W.T @ (x - b) / s ** 2                        # posterior mean
print(f"log p(x), exact: {log_px:.4f}")
print(f"true posterior: mean ({m[0]:.3f}, {m[1]:.3f}), "
      f"std ({np.sqrt(P[0, 0]):.3f}, {np.sqrt(P[1, 1]):.3f}), correlation {P[0, 1] / np.sqrt(P[0, 0] * P[1, 1]):.3f}\n")
print(f"{'q(z|x)':36s} {'recon (sampled, exact)':>22s} {'KL to prior':>11s} {'ELBO':>8s} {'gap':>7s} {'KL to post':>10s}")
# The best diagonal q (smallest KL to the posterior) keeps the posterior's
# mean but takes std_j = 1 / sqrt(precision_jj), not the posterior's own
# marginal stds: Kingma and Welling's encoder can't do better than this.
best = 1 / np.sqrt(np.diag(np.linalg.inv(P)))
cases = [("best diagonal Gaussian", m, best),
         ("posterior's mean and marginal stds", m, np.sqrt(np.diag(P))),
         ("right mean, stds 3x too wide", m, 3 * np.sqrt(np.diag(P))),
         ("the prior, N(0, I)", np.zeros(2), np.ones(2))]
for name, mu, sigma in cases:
    rec, exact, kl = elbo(mu, sigma)
    print(f"{name:36s} {rec:10.4f} {exact:10.4f}  {kl:11.4f} {exact - kl:8.4f} {log_px - (exact - kl):7.4f} "
          f"{kl_gauss(mu, np.diag(sigma ** 2), m, P):10.4f}")

mu, sigma = np.array([1.0, -0.5]), np.array([0.5, 2.0])
print(f"\nKL(N(mu, sigma^2) || N(0, I)) for mu = (1, -0.5), sigma = (0.5, 2): "
      f"{0.5 * (mu ** 2 + sigma ** 2 - 1 - np.log(sigma ** 2)).sum():.4f} nats")

The post printed:

```text
log p(x), exact: -2.3429
true posterior: mean (0.365, 0.618), std (0.442, 0.479), correlation -0.210

q(z|x)                               recon (sampled, exact) KL to prior     ELBO     gap KL to post
best diagonal Gaussian                  -1.3081    -1.3066       1.0590  -2.3655  0.0226     0.0226
posterior's mean and marginal stds      -1.3417    -1.3435       1.0231  -2.3666  0.0237     0.0237
right mean, stds 3x too wide            -8.0050    -8.0153       0.5245  -8.5398  6.1969     6.1969
the prior, N(0, I)                      -6.1847    -6.1916       0.0000  -6.1916  3.8487     3.8487

KL(N(mu, sigma^2) || N(0, I)) for mu = (1, -0.5), sigma = (0.5, 2): 1.7500 nats
```

## Three models, one test

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/vaes-and-gans/#three-models-one-test)

In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn

# Three generative models on the same 2-D data: 8 Gaussian modes on a ring
# of radius 2 (std 0.05 each), the standard test for mode collapse. Each is
# scored on 2,000 samples: modes covered (a mode counts if at least 1% of
# samples fall within 3 std of it) and quality (the share of samples within
# 3 std of some mode). Three seeds each.
centers = torch.tensor([[2 * math.cos(a), 2 * math.sin(a)] for a in np.arange(8) * math.pi / 4], dtype=torch.float32)
def data(n, g):
    return centers[torch.randint(0, 8, (n,), generator=g)] + 0.05 * torch.randn(n, 2, generator=g)
def score(x):
    d = torch.cdist(x, centers)
    near = d.min(1).values < 0.15
    hits = torch.bincount(d.argmin(1)[near], minlength=8)
    return int((hits > 0.01 * len(x)).sum()), near.float().mean().item()
mlp = lambda i, o: nn.Sequential(nn.Linear(i, 128), nn.ReLU(), nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, o))

def vae(seed, steps=6000, beta=1.0):
    """Encoder gives q(z|x) = N(mu, sigma^2) in 2-D; decoder gives p(x|z) =
    N(f(z), 0.05^2 I). Loss: minus the ELBO = reconstruction + KL to N(0, I)."""
    g = torch.Generator().manual_seed(seed); torch.manual_seed(seed)
    enc, dec = mlp(2, 4), mlp(2, 2)
    opt = torch.optim.Adam([*enc.parameters(), *dec.parameters()], 1e-3)
    for _ in range(steps):
        x = data(256, g)
        mu, logvar = enc(x).chunk(2, -1)
        z = mu + torch.randn_like(mu) * (0.5 * logvar).exp()            # the reparameterization trick
        rec = ((dec(z) - x) ** 2).sum(-1) / (2 * 0.05 ** 2)
        kl = 0.5 * (mu ** 2 + logvar.exp() - 1 - logvar).sum(-1)
        loss = (rec + beta * kl).mean()
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        return score(dec(torch.randn(2000, 2)))

def gan(seed, steps=6000, wgan=False):
    """Generator maps 2-D noise to points; the critic tells data from
    samples. Ordinary GAN: the non-saturating loss of Goodfellow et al.
    WGAN-GP: a critic score difference with a gradient penalty."""
    g = torch.Generator().manual_seed(seed); torch.manual_seed(seed)
    G, D = mlp(2, 2), mlp(2, 1)
    og = torch.optim.Adam(G.parameters(), 1e-4 if wgan else 2e-4, betas=(0.5, 0.9))
    od = torch.optim.Adam(D.parameters(), 1e-4 if wgan else 2e-4, betas=(0.5, 0.9))
    bce = nn.BCEWithLogitsLoss()
    for step in range(steps):
        for _ in range(5 if wgan else 1):
            x, fake = data(256, g), G(torch.randn(256, 2)).detach()
            if wgan:
                eps = torch.rand(256, 1)
                mix = (eps * x + (1 - eps) * fake).requires_grad_(True)
                grad = torch.autograd.grad(D(mix).sum(), mix, create_graph=True)[0]
                ld = D(fake).mean() - D(x).mean() + 10 * ((grad.norm(dim=1) - 1) ** 2).mean()
            else:
                ld = bce(D(x), torch.ones(256, 1)) + bce(D(fake), torch.zeros(256, 1))
            od.zero_grad(); ld.backward(); od.step()
        fake = G(torch.randn(256, 2))
        lg = -D(fake).mean() if wgan else bce(D(fake), torch.ones(256, 1))
        og.zero_grad(); lg.backward(); og.step()
    with torch.no_grad():
        return score(G(torch.randn(2000, 2)))

# The GAN's value at the optimal discriminator is 2 JS(p, q) - log 4: checked
# for two 1-D Gaussians on a grid.
xs = np.linspace(-10, 10, 20001); dx = xs[1] - xs[0]
p = np.exp(-0.5 * xs ** 2) / math.sqrt(2 * math.pi); q = np.exp(-0.5 * (xs - 1.5) ** 2) / math.sqrt(2 * math.pi)
Dstar = p / (p + q)
value = (p * np.log(Dstar) + q * np.log(1 - Dstar)).sum() * dx
m = (p + q) / 2
js = 0.5 * ((p * np.log(p / m)).sum() + (q * np.log(q / m)).sum()) * dx
print(f"optimal-discriminator value {value:.4f}; 2 JS - log 4 = {2 * js - math.log(4):.4f}")

print(f"\n{'model':24s} modes covered (of 8), per seed   share of samples near a mode")
for name, fn in (("VAE", vae), ("GAN (non-saturating)", gan), ("WGAN-GP", lambda s: gan(s, wgan=True))):
    res = [fn(s) for s in range(3)]
    print(f"{name:24s} {'  '.join(str(r[0]) for r in res):>18s}            {'  '.join(f'{r[1]:.0%}' for r in res)}", flush=True)

The post printed:

```text
model                    modes covered (of 8), per seed   share of samples near a mode
VAE                                 8  8  8            52%  48%  51%
GAN (non-saturating)                8  7  8            89%  82%  88%
WGAN-GP                             8  8  1            72%  65%  2%
```

In [ ]:
import torch
# Where do the VAE's off-mode samples come from? gen2d.py's vae(), trained
# the same way, but returning the encoder and decoder instead of a score.
src = open("gen2d.py").read().split("# The GAN's value")[0]
ns = {}; exec(src.replace("return score(dec(torch.randn(2000, 2)))", "return enc, dec"), ns)
print(f"{'seed':>4s} {'reconstructions':>16s} {'z from the encoders':>20s} {'z from the prior':>17s}")
for seed in range(3):
    enc, dec = ns["vae"](seed)
    with torch.no_grad():
        x = ns["data"](2000, torch.Generator().manual_seed(99))
        mu, logvar = enc(x).chunk(2, -1)
        rec = dec(mu)                                                    # decode each point's mean code
        agg = dec(mu + torch.randn_like(mu) * (0.5 * logvar).exp())      # a code the encoder could give
        prior = dec(torch.randn(2000, 2))                                # what sampling the VAE does
    print(f"{seed:4d} " + "".join(f"{ns['score'](v)[1]:>{w}.0%}" for v, w in ((rec, 16), (agg, 21), (prior, 18))), flush=True)

The post printed:

```text
seed  reconstructions  z from the encoders  z from the prior
   0             100%                 100%               52%
   1             100%                  99%               50%
   2             100%                 100%               48%
```

In [ ]:
import torch
# WGAN-GP, seed 2, run to 12,000 generator steps and scored every 1,500:
# gen2d.py's gan() with a scoring hook. Scoring draws from the same global
# random generator as the generator's noise, so after the first checkpoint
# this run follows a different path from the one in the table.
src = open("gen2d.py").read().split("# The GAN's value")[0]
src = src.replace("        og.zero_grad(); lg.backward(); og.step()\n",
                  "        og.zero_grad(); lg.backward(); og.step()\n"
                  "        if (step + 1) % 1500 == 0: report(step + 1, G)\n")
ns = {}; exec(src, ns)
def report(step, G):
    with torch.no_grad():
        x = G(torch.randn(2000, 2))
    modes, near = ns["score"](x)
    print(f"step {step:6d}   modes {modes}   near a mode {near:5.1%}   "
          f"sample std ({x[:, 0].std():.2f}, {x[:, 1].std():.2f})", flush=True)
ns["report"] = report
ns["gan"](2, steps=12000, wgan=True)

The post printed:

```text
step   1500   modes 0   near a mode  0.1%   sample std (0.69, 0.43)
step   3000   modes 0   near a mode  0.8%   sample std (3.05, 2.17)
step   4500   modes 0   near a mode  0.0%   sample std (0.16, 0.24)
step   6000   modes 0   near a mode  0.0%   sample std (0.27, 0.63)
step   7500   modes 0   near a mode  1.0%   sample std (0.99, 2.04)
step   9000   modes 0   near a mode  0.2%   sample std (1.05, 1.03)
step  10500   modes 0   near a mode  0.4%   sample std (0.44, 0.83)
step  12000   modes 0   near a mode  1.4%   sample std (0.89, 1.00)
```